# Lab 2 — Number Detection (YOLOv8)

## §1. Clone / Pull + path setup

In [ ]:
import os
import sys
from pathlib import Path

if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import userdata  # type: ignore
    github_token = userdata.get('GITHUB_TOKEN')
    repo_url = f"https://{github_token}@github.com/Ma-XD/ITMO-CV.git"
    
    if not Path("/content/ITMO-CV").exists():
        !git clone {repo_url} /content/ITMO-CV
    else:
        %cd /content/ITMO-CV
        !git pull
    
    sys.path.insert(0, "/content/ITMO-CV/lab2-DET")
    %cd /content/ITMO-CV/lab2-DET
else:
    LAB_DIR = Path(".").resolve()
    if LAB_DIR.name != "lab2-DET":
        %cd lab2-DET
    if str(LAB_DIR) not in sys.path:
        sys.path.insert(0, str(LAB_DIR))

## §2. Drive mount

In [ ]:
from env_config import is_colab

if is_colab:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')

## §3. Зависимости

In [ ]:
!pip install -q -r requirements.txt

## §4. Verify окружения

In [ ]:
from env_config import print_env
print_env()

## §5. Загрузка датасета
Копируем заранее подготовленные архивы с Google Drive и распаковываем их в локальное хранилище Colab (`/content/data/`).

In [ ]:
from env_config import is_colab

if is_colab:
    !mkdir -p /content/data
    
    print("📦 Копирование SVHN...")
    !cp "/content/drive/MyDrive/ITMO-CV/lab2-DET/data/svhn_yolo.zip" /content/data/
    !unzip -q /content/data/svhn_yolo.zip -d /content/data/svhn_yolo
    
    print("📦 Копирование Toronto Number Detection...")
    !cp "/content/drive/MyDrive/ITMO-CV/lab2-DET/data/numberdetection.v1i.yolov8.zip" /content/data/
    !unzip -q /content/data/numberdetection.v1i.yolov8.zip -d /content/data/numberdetection
    
    print("✅ Данные успешно скопированы и распакованы в /content/data/")
else:
    print("💻 Локальный запуск: используем данные из lab2-DET/data/")

## §6. Подготовка разметки (YOLO format)

Исходный датасет SVHN поставляется в формате `digitStruct.mat` (HDF5). Для работы с YOLOv8 я предварительно сконвертировал его в нужный формат с помощью скрипта `scripts/build_dataset.py`.

Скрипт выполнил:
1. Парсинг `digitStruct.mat` и извлечение координат боксов (top, left, height, width).
2. Перевод координат в нормализованный формат YOLO (cx, cy, w, h).
3. Разделение на `train` (90%) и `val` (10%).
4. Создание файла `data.yaml`.

В Colab этот шаг пропускается, так как уже загружен готовый архив `svhn_yolo.zip`.

## §7. Работа с датасетом
Проверим корректность разметки: загрузим одну картинку из SVHN и нарисуем поверх неё боксы из соответствующего `.txt` файла.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from config import SVHN_DIR

def plot_yolo_sample(img_path: Path, label_path: Path):
    img = Image.open(img_path)
    fig, ax = plt.subplots(1, figsize=(6, 6))
    ax.imshow(img)
    
    if label_path.exists():
        with open(label_path, "r") as f:
            lines = f.readlines()
        
        w, h = img.size
        for line in lines:
            class_id, cx, cy, bw, bh = map(float, line.strip().split())
            # YOLO format (cx, cy, bw, bh) -> Matplotlib format (x_min, y_min, width, height)
            box_w = bw * w
            box_h = bh * h
            x_min = (cx * w) - (box_w / 2)
            y_min = (cy * h) - (box_h / 2)
            
            rect = patches.Rectangle((x_min, y_min), box_w, box_h, linewidth=2, edgecolor='r', facecolor='none')
            ax.add_patch(rect)
            ax.text(x_min, y_min - 2, str(int(class_id)), color='red', fontsize=12, weight='bold')
    else:
        print("⚠️ Файл разметки не найден!")
        
    plt.axis('off')
    plt.show()

# Визуализируем один пример из обучающей выборки SVHN
sample_img = SVHN_DIR / "images/train/1.png"
sample_lbl = SVHN_DIR / "labels/train/1.txt"

if sample_img.exists():
    print(f"Визуализация: {sample_img.name}")
    plot_yolo_sample(sample_img, sample_lbl)
else:
    print("⚠️ Изображение не найдено. Проверьте пути.")